# Evaluacion del modelo
* Pérdida y Precisión Global.
* Precision, Recall y F1-Score por cada una de las 200 especies.
* Identificación de las especies con mejor y peor rendimiento (cuellos de botella del modelo).
* Visualización de predicciones correctas y errores cometidos por la red.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
# 🌟 ACTIVAMOS LA PRECISION MIXTA AQUÍ TAMBIÉN
from tensorflow.keras import mixed_precision

# Parche de memoria por si ejecutas este notebook de forma independiente
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# Configuración de dimensiones
IMG_ROWS, IMG_COLS = 256, 256
BATCH_SIZE = 32
data_dir = 'CUB_200_2011/images'

I0000 00:00:1778949805.071708   64496 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Entorno sintonizado en float16. Versión de TensorFlow: 2.21.0


In [2]:
# Cargamos el dataset de validación de la misma forma exacta que en el entrenamiento
validation_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_ROWS, IMG_COLS),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Extraemos los nombres de las clases antes de optimizar el dataset
class_names = validation_ds.class_names

# Normalización idéntica a la de entrenamiento
normalization_layer = tf.keras.layers.Rescaling(1./255)
validation_ds = validation_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)

# Prefetch asíncrono para velocidad de lectura
validation_ds = validation_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

print(f"Dataset de validación cargado con {len(class_names)} clases.")

Found 11788 files belonging to 200 classes.
Using 2357 files for validation.


I0000 00:00:1778949808.879542   64496 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1778949808.879869   64496 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:26:00.0, compute capability: 8.6


Dataset de validación cargado con 200 clases.


In [3]:
# Carga aquí el archivo .keras que hayas guardado al terminar el entrenamiento
# Nota: Si el modelo está todavía en la memoria de Jupyter de tu otro notebook, puedes saltarte este paso.
from tensorflow.keras.models import load_model

model_path = "birds_reentrenado.keras"  # Ajusta el nombre al de tu archivo guardado
if os.path.exists(model_path):
    model = load_model(model_path)
    print("¡Modelo cargado correctamente desde el disco!")
else:
    print(f"No se encontró el archivo en {model_path}. Asegúrate de haber guardado el modelo primero.")

¡Modelo cargado correctamente desde el disco!


In [4]:
print("Calculando métricas globales en el set de validación...")
loss, accuracy = model.evaluate(validation_ds, verbose=1)

print("\n==========================================")
print(f"Pérdida Global (Loss): {loss:.4f}")
print(f"Precisión Global (Accuracy): {accuracy * 100:.2f}%")
print("==========================================")

Calculando métricas globales en el set de validación...


I0000 00:00:1778949812.820550   85192 service.cc:153] XLA service 0x7f6bd025d9c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778949812.820599   85192 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Ti, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1778949812.896565   85192 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778949813.246198   85192 cuda_dnn.cc:461] Loaded cuDNN version 92200


 5/74 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.6425 - loss: 1.4036

I0000 00:00:1778949820.944893   85192 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


74/74 ━━━━━━━━━━━━━━━━━━━━ 16s 86ms/step - accuracy: 0.6258 - loss: 1.4670

Pérdida Global (Loss): 1.4670
Precisión Global (Accuracy): 62.58%


In [5]:
print("Generando predicciones detalladas lote por lote de forma correcta...")

y_true = []
y_pred = []

# Iteramos de forma eficiente sobre el dataset
for images, labels in validation_ds:
    # 🌟 SOLUCIÓN: Llamamos al modelo directamente como función. 
    # Esto evita el bug de desbordamiento de Softmax en float16.
    preds = model(images, training=False)
    
    # Convertimos los tensores a arrays de NumPy y extraemos el índice más alto
    y_pred.extend(np.argmax(preds.numpy(), axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(f"Predicciones completadas con éxito para las {len(y_true)} imágenes.")

Generando predicciones detalladas lote por lote de forma correcta...
Predicciones completadas con éxito para las 2357 imágenes.


In [6]:
# Generamos el reporte completo de Scikit-Learn
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

# Mostramos el resumen macro y weighted promedio
print("--- RESUMEN DE MÉTRICAS PROMEDIO ---")
print(f"Precision Promedio (Weighted): {report_dict['weighted avg']['precision']*100:.2f}%")
print(f"Recall Promedio (Weighted): {report_dict['weighted avg']['recall']*100:.2f}%")
print(f"F1-Score Promedio (Weighted): {report_dict['weighted avg']['f1-score']*100:.2f}%")

# Extraemos el rendimiento individual para ordenarlos
class_performance = []
for class_name in class_names:
    class_performance.append({
        'clase': class_name,
        'f1-score': report_dict[class_name]['f1-score'],
        'precision': report_dict[class_name]['precision'],
        'recall': report_dict[class_name]['recall']
    })

--- RESUMEN DE MÉTRICAS PROMEDIO ---
Precision Promedio (Weighted): 65.51%
Recall Promedio (Weighted): 62.49%
F1-Score Promedio (Weighted): 62.10%


In [7]:
# Ordenamos las clases según su F1-Score
class_performance_sorted = sorted(class_performance, key=lambda x: x['f1-score'], reverse=True)

print("=========================================================")
print("🏆 TOP 10 ESPECIES CON MEJOR RENDIMIENTO (F1-SCORE) 🏆")
print("=========================================================")
for i in range(10):
    c = class_performance_sorted[i]
    print(f"{i+1}. {c['clase']}: F1={c['f1-score']*100:.1f}% | Prec={c['precision']*100:.1f}% | Rec={c['recall']*100:.1f}%")

print("\n=========================================================")
print("🛑 TOP 10 ESPECIES CON PEOR RENDIMIENTO (CUELLOS DE BOTELLA) 🛑")
print("=========================================================")
for i in range(1, 11):
    c = class_performance_sorted[-i]
    print(f"{i}. {c['clase']}: F1={c['f1-score']*100:.1f}% | Prec={c['precision']*100:.1f}% | Rec={c['recall']*100:.1f}%")

🏆 TOP 10 ESPECIES CON MEJOR RENDIMIENTO (F1-SCORE) 🏆
1. 018.Spotted_Catbird: F1=100.0% | Prec=100.0% | Rec=100.0%
2. 075.Green_Jay: F1=100.0% | Prec=100.0% | Rec=100.0%
3. 191.Red_headed_Woodpecker: F1=97.1% | Prec=100.0% | Rec=94.4%
4. 012.Yellow_headed_Blackbird: F1=96.0% | Prec=100.0% | Rec=92.3%
5. 106.Horned_Puffin: F1=96.0% | Prec=92.3% | Rec=100.0%
6. 053.Western_Grebe: F1=95.2% | Prec=100.0% | Rec=90.9%
7. 070.Green_Violetear: F1=95.0% | Prec=95.0% | Rec=95.0%
8. 007.Parakeet_Auklet: F1=94.1% | Prec=100.0% | Rec=88.9%
9. 044.Frigatebird: F1=93.3% | Prec=100.0% | Rec=87.5%
10. 035.Purple_Finch: F1=92.3% | Prec=100.0% | Rec=85.7%

🛑 TOP 10 ESPECIES CON PEOR RENDIMIENTO (CUELLOS DE BOTELLA) 🛑
1. 116.Chipping_Sparrow: F1=7.4% | Prec=7.7% | Rec=7.1%
2. 043.Yellow_bellied_Flycatcher: F1=8.0% | Prec=7.1% | Rec=9.1%
3. 030.Fish_Crow: F1=8.7% | Prec=12.5% | Rec=6.7%
4. 121.Grasshopper_Sparrow: F1=10.5% | Prec=33.3% | Rec=6.2%
5. 129.Song_Sparrow: F1=11.1% | Prec=10.0% | Rec=12.5%
6. 037

In [ ]:
# Extraemos un único batch del dataset para pintar ejemplos reales en pantalla
images, labels = next(iter(validation_ds))
preds = model.predict(images, verbose=0)

plt.figure(figsize=(16, 12))
for i in range(12): # Pintamos las primeras 12 imágenes del lote
    plt.subplot(3, 4, i + 1)
    
    # 🌟 SOLUCIÓN: Convertimos explícitamente el array a float32 para que Matplotlib sepa pintarlo
    img = images[i].numpy().astype('float32')
    
    true_idx = np.argmax(labels[i])
    pred_idx = np.argmax(preds[i])
    prob = preds[i][pred_idx] * 100
    
    plt.imshow(img)
    
    # Si acertó pintamos las letras en verde, si falló en rojo mostrando el error
    if true_idx == pred_idx:
        title_color = 'green'
        title_text = f"OK: {class_names[pred_idx]}\n({prob:.1f}%)"
    else:
        title_color = 'red'
        title_text = f"PRED: {class_names[pred_idx]} ({prob:.1f}%)\nREAL: {class_names[true_idx]}"
        
    plt.title(title_text, color=title_color, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

## 6. Análisis forense SHAP

¿En qué se fija la IA para decir que un 7 es un 7 (o en nuestro caso, una especie u otra)? SHAP crea mapas de calor donde los píxeles rojos "apoyan" la decisión y los azules "la contradicen". Es la mejor forma de saber si la IA es inteligente o si está mirando el fondo de la imagen por error.

**Compara el mapa SHAP** de un modelo entrenado poco tiempo vs uno con muchas épocas.
**Pregunta**: ¿En cuál de los dos ves áreas de color más definidas sobre el animal? (Un modelo malo suele ver "fantasmas" o ruido en el fondo blanco).

In [ ]:
import shap
import numpy as np
import tensorflow as tf
import warnings

warnings.filterwarnings("ignore")

IMG_ROWS = 256
IMG_COLS = 256
CHANNELS = 3
IMG_SHAPE = (IMG_ROWS, IMG_COLS, CHANNELS)

if 'validation_generator' in locals() or 'validation_generator' in globals():
    images_batch, labels_batch = next(validation_ds)
    x_test_exp = images_batch.astype('float32')
    y_test = np.argmax(labels_batch, axis=1)
else:
    images_batch, labels_batch = next(iter(validation_ds))
    x_test_exp = images_batch.numpy().astype('float32')
    y_test = np.argmax(labels_batch.numpy(), axis=1)

idx = 0
target_class = y_test[idx]

logits_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[-2].output)

# Usamos un fondo pequeño pero limpio en float32
background = x_test_exp[:2].astype(np.float32)

# Creamos el entorno de entrada explícito
input_tensor = tf.keras.Input(shape=IMG_SHAPE)
output_tensor = logits_model(input_tensor)

explainer = shap.GradientExplainer(
    (input_tensor, output_tensor[:, int(target_class)]),
    background
)

# Calculamos los valores SHAP sobre los logits
shap_values = explainer.shap_values(x_test_exp[idx:idx+1])

shap_values_limpios = np.squeeze(np.array(shap_values))
if shap_values_limpios.ndim == 3:
    shap_values_limpios = np.expand_dims(shap_values_limpios, axis=0)

imagen_float32 = x_test_exp[idx:idx+1].astype('float32')

print(f"Clase real de ave analizada (índice): {target_class}")

shap.image_plot([shap_values_limpios], imagen_float32)

## 7. Buscando errores en el modelo

Para mejorar una IA, debemos ser detectives. En esta sección automatizamos la búsqueda de fallos específicos (Falsos Positivos y Falsos Negativos) y le pedimos a SHAP que nos explique por qué la IA se equivocó en esos casos concretos.

**Pregunta**: Mira un ejemplo donde la IA falló. ¿Es una imagen donde al propio humano le costaría identificar al pájaro (camuflado o cortado), o es un error tonto de la IA?

In [ ]:
import shap
import numpy as np
import tensorflow as tf
import warnings

warnings.filterwarnings("ignore")

# 1. Interceptamos los LOGITS (la capa Dense antes del Softmax)
logits_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[-2].output)

# 2. Obtenemos las predicciones del batch evitando el bug de predict()
preds_probs = model(x_test_exp, training=False)
preds = np.argmax(preds_probs.numpy(), axis=1)

# Detectar imágenes dentro del batch donde la IA falló
idx_errores = np.where(y_test != preds)[0]

if len(idx_errores) > 0:
    sel = idx_errores[:2] # Ver solo 2 ejemplos de error para ir rápido
    
    for i, s in enumerate(sel):
        clase_predicha = preds[s]
        clase_real = y_test[s]
        
        # 🌟 TRADUCCIÓN: Extraemos el nombre de la especie usando class_names
        nombre_real = class_names[clase_real]
        nombre_predicho = class_names[clase_predicha]
        
        # 🌟 MENSAJE MEJORADO: Ahora muestra "Número.Nombre_de_la_Especie"
        print(f"\n Error {i+1}:{nombre_real}")
        print(f" La IA se confundió y la clasificó como {nombre_predicho}")
        
        # Fondo limpio en float32 para la comparación de SHAP
        background = x_test_exp[:2].astype(np.float32)
        
        # Entorno de entrada explícito usando el submodelo de Logits
        input_tensor = tf.keras.Input(shape=(256, 256, 3))
        output_tensor = logits_model(input_tensor)
        
        # Evaluamos la neurona de la clase incorrecta (el Falso Positivo)
        explainer_err = shap.GradientExplainer(
            (input_tensor, output_tensor[:, int(clase_predicha)]),
            background
        )
        
        shap_vals_err = explainer_err.shap_values(x_test_exp[s:s+1])
        
        # Limpieza de dimensiones para forzar 4D estricto
        shap_vals_err_limpios = np.squeeze(np.array(shap_vals_err))
        if shap_vals_err_limpios.ndim == 3:
            shap_vals_err_limpios = np.expand_dims(shap_vals_err_limpios, axis=0)
            
        imagen_float32_err = x_test_exp[s:s+1].astype('float32')
            
        # Pintamos la máscara final
        shap.image_plot([shap_vals_err_limpios], imagen_float32_err)
else:
    print("¡No hubo errores en este batch de test!")

## 8. Confianza

En aplicaciones críticas (como medicina o banca), no basta con que la IA responda; necesitamos que nos diga qué tan segura está. Esta gráfica muestra cómo, al exigir mayor seguridad (umbral), eliminamos errores, pero a cambio necesitamos que un humano revise más casos ("Descartes").

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("Calculando análisis de confianza alineando predicciones y etiquetas reales...")

all_probs = []
all_true = []

# 🚀 GARANTÍA DE CONFIANZA: Extraemos todo en el mismo bucle para evitar desalineaciones
for images, labels in validation_ds:
    preds = model(images, training=False)
    all_probs.extend(preds.numpy())
    all_true.extend(np.argmax(labels.numpy(), axis=1))

all_probs = np.array(all_probs)
all_true = np.array(all_true)

# Calculamos confianzas máximas y aciertos absolutos
conf = np.max(all_probs, axis=1)
preds_totales = np.argmax(all_probs, axis=1)
correctos = (preds_totales == all_true)

# 2. Cálculo de métricas por umbral de confianza
umbrales = [0.80, 0.70, 0.50, 0.20, 0.00]
res = []
total_m = len(conf)

for u in umbrales:
    pasan = conf >= u
    n = np.sum(pasan)
    
    aciertos = np.sum(correctos[pasan]) if n > 0 else 0
    errores = n - aciertos
    duda = total_m - n
    
    # Porcentajes respecto al TOTAL del dataset
    pct_aciertos = (aciertos / total_m) * 100
    pct_errores = (errores / total_m) * 100
    pct_duda = (duda / total_m) * 100
    
    # Precisión quirúrgica: de las que la IA se atrevió a clasificar, cuántas acertó
    precision_ia = (aciertos / n) * 100 if n > 0 else 100.0
    
    res.append([u * 100, pct_aciertos, pct_errores, pct_duda, precision_ia])

# Creamos el DataFrame explicativo
df = pd.DataFrame(res, columns=['Umbral Confianza (%)', 'Aciertos Automatizados (%)', 'Errores Automatizados (%)', 'Derivado a Humano (%)', 'Precisión de la IA (%)'])

# Mostramos la tabla por consola para poder copiar los datos numéricos a la memoria
print("\n--- TABLA DE DECISIÓN ESTRATÉGICA ---")
print(df.to_string(index=False, formatters={
    'Umbral Confianza (%)': '{:.0f}%'.format,
    'Aciertos Automatizados (%)': '{:.2f}%'.format,
    'Errores Automatizados (%)': '{:.2f}%'.format,
    'Derivado a Humano (%)': '{:.2f}%'.format,
    'Precisión de la IA (%)': '{:.2f}%'.format
}))

# 3. Gráfica con acabado profesional
fig, ax1 = plt.subplots(figsize=(11, 6))
x = range(len(df))

# Colores elegantes y armónicos (Estilo Flat Design)
ax1.bar(x, df['Aciertos Automatizados (%)'], color='#2ecc71', alpha=0.85, label='Aciertos IA', width=0.5)
ax1.bar(x, df['Errores Automatizados (%)'], bottom=df['Aciertos Automatizados (%)'], color='#e74c3c', alpha=0.85, label='Errores IA', width=0.5)
ax1.bar(x, df['Derivado a Humano (%)'], bottom=df['Aciertos Automatizados (%)'] + df['Errores Automatizados (%)'], color='#b2bec3', alpha=0.4, label='Duda (Revisión Humana)', width=0.5)

# Configuración del eje primario (Barras)
ax1.set_ylabel('Porcentaje sobre el Total de Imágenes (%)', fontsize=11, fontweight='bold', color='#2d3436')
ax1.set_xlabel('Umbral de Certeza Mínimo exigido a la IA', fontsize=11, fontweight='bold', color='#2d3436')
ax1.set_xticks(x)
ax1.set_xticklabels([f"≥ {int(i)}%" if i > 0 else "Sin Filtro" for i in df['Umbral Confianza (%)']], fontsize=10)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# Eje secundario (Línea de calidad)
ax2 = ax1.twinx()
ax2.plot(x, df['Precisión de la IA (%)'], color='#0984e3', linewidth=3, marker='o', markersize=8, label='Precisión Real de la IA')
ax2.set_ylabel('Precisión Interna de los Aciertos (%)', fontsize=11, fontweight='bold', color='#0984e3')
ax2.tick_params(axis='y', labelcolor='#0984e3')
ax2.set_ylim(0, 105)

# Título y Leyendas fusionadas
plt.title("Control de Riesgo: Precisión de la IA vs Tasa de Automatización", fontsize=14, fontweight='bold', pad=20, color='#2d3436')
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', bbox_to_anchor=(1.12, 1), frameon=True, facecolor='white')

plt.tight_layout()
plt.show()